# CaBU: Calibrated, Boundary-aware Unlearning for tumour segmentation (BCSS)

This notebook runs every experiment in the CaBU article and writes the tables and figures for its Results section.

**Before you run**
1. Add the BCSS dataset as input (the Kaggle BCSS 512 dataset you already use works). The code finds the folder by itself: masks must sit in a folder whose name contains `mask`, with the same file names as the images, and the file names must contain the TCGA barcode (for example `TCGA-A2-A0CM`).
2. Settings → Accelerator → **GPU T4 x2** or **P100**. Internet can stay off.
3. Use **Save Version → Save & Run All (Commit)** so the run continues in the background (up to 12 h) and the outputs are kept.

**Two sessions are recommended**
- Session 1: `RUN_PREPARE`, `RUN_S1` → about 3–5 h on a T4 (1 original model, 7 retrained models, 14 unlearning variants per setting).
- Session 2: add the Session 1 output as an input, set `RESUME_DIR` to its `cabu_run` folder, then `RUN_S2` and `RUN_REPORT` → about 4–6 h (3 corrupted models and 12 retrained models).

Every finished model and every finished result row is saved, so a stopped run resumes where it left off.
At the end, download **`cabu_results.zip`** and send it back for the Results section.

In [1]:
import os, re, glob, shutil, subprocess

# ---- settings -------------------------------------------------------------------------
RUN_PREPARE = True
RUN_S1      = True
RUN_S2      = False      # set True in session 2
RUN_REPORT  = True
RESUME_DIR  = ''         # session 2: e.g. '/kaggle/input/<your-session-1-notebook>/cabu_run'
WORK        = '/kaggle/working/cabu_run'
EXTRA       = ''         # e.g. '--epochs 25 --seeds 0 1' to shorten the run, '--arch unet' for the second backbone

# ---- find the BCSS folder under /kaggle/input --------------------------------------------
cands = []
for dp, dn, fn in os.walk('/kaggle/input'):
    if 'cabu_run' in dp:
        continue
    if any(re.search(r'TCGA-[A-Z0-9]{2}-[A-Z0-9]{4}', f, re.I) for f in fn[:200]):
        cands.append(dp)
DATA_ROOT = os.path.commonpath(cands) if cands else '/kaggle/input'
print('BCSS folders found:', cands[:6])
print('DATA_ROOT =', DATA_ROOT)
!nvidia-smi --query-gpu=name,memory.total --format=csv

BCSS folders found: ['/kaggle/input/datasets/whats2000/breast-cancer-semantic-segmentation-bcss/BCSS/test', '/kaggle/input/datasets/whats2000/breast-cancer-semantic-segmentation-bcss/BCSS/train', '/kaggle/input/datasets/whats2000/breast-cancer-semantic-segmentation-bcss/BCSS/train_mask', '/kaggle/input/datasets/whats2000/breast-cancer-semantic-segmentation-bcss/BCSS/val', '/kaggle/input/datasets/whats2000/breast-cancer-semantic-segmentation-bcss/BCSS/val_mask', '/kaggle/input/datasets/whats2000/breast-cancer-semantic-segmentation-bcss/BCSS_512/train_512']
DATA_ROOT = /kaggle/input/datasets/whats2000/breast-cancer-semantic-segmentation-bcss
name, memory.total [MiB]
Tesla T4, 15360 MiB
Tesla T4, 15360 MiB


In [2]:
%%writefile cabu.py
#!/usr/bin/env python3
"""CaBU: Calibrated, Boundary-aware Unlearning for tumour segmentation on BCSS.

Stages
  prepare  find BCSS image/mask pairs, map labels to 5 classes, tile, resize, cache, make patient-level splits
  s1       scenario S1: patient (S1-P) and institution (S1-I) withdrawal, all methods vs retraining
  s2       scenario S2: corrective unlearning of corrupted tumour annotations with partial identification
  report   aggregate results.csv into tables (markdown + csv) and figures
  smoke    build a small synthetic BCSS-like dataset and run every stage end to end (for testing)

Every expensive step is checkpointed in --work, so a stage can be re-run to resume after a session ends.
"""
import argparse, copy, csv, json, math, os, random, re, shutil, sys, time
from collections import defaultdict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from scipy import ndimage
from scipy.stats import ks_2samp

Image.MAX_IMAGE_PIXELS = None

CLASSES = ['tumour', 'stroma', 'inflammatory', 'necrosis', 'other']
NC, IGN = 5, 255
TUM, STR, INF, NEC, OTH = range(5)
DEV = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
CUDA = DEV.type == 'cuda'


# ----------------------------------------------------------------------------- utilities
def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if CUDA:
        torch.cuda.manual_seed_all(s)


def autocast():
    return torch.autocast(device_type='cuda', dtype=torch.float16, enabled=CUDA)


def grad_scaler():
    try:
        return torch.amp.GradScaler('cuda', enabled=CUDA)
    except Exception:  # older torch
        return torch.cuda.amp.GradScaler(enabled=CUDA)


def jdump(obj, path):
    with open(path, 'w') as f:
        json.dump(obj, f, indent=1)


def jload(path):
    with open(path) as f:
        return json.load(f)


def bcss_lut():
    """BCSS label codes (0-21) -> 5 classes; outside_roi (0) and exclude (7) -> ignore."""
    lut = np.full(256, OTH, np.uint8)
    lut[0] = IGN; lut[7] = IGN
    lut[1] = TUM
    lut[2] = STR
    lut[3] = INF; lut[10] = INF; lut[11] = INF   # lymphocytic, plasma cells, other immune infiltrate
    lut[4] = NEC
    lut[22:] = IGN                              # unknown codes
    return lut


# ----------------------------------------------------------------------------- data discovery and caching
PAT_RE = re.compile(r'TCGA-([A-Z0-9]{2})-([A-Z0-9]{4})', re.I)
IMG_EXT = ('.png', '.jpg', '.jpeg', '.tif', '.tiff')
MASK_DIR_WORDS = ('mask', 'gt', 'label', 'annotation')


def _is_mask_path(path):
    parents = [p.lower() for p in path.split(os.sep)[-3:-1]]
    stem = os.path.splitext(os.path.basename(path))[0].lower()
    if any(w in p for p in parents for w in MASK_DIR_WORDS):
        return True
    return bool(re.search(r'[_-](mask|gt|label)$', stem))


def _key(path):
    stem = os.path.splitext(os.path.basename(path))[0]
    return re.sub(r'[_-](mask|gt|label)$', '', stem, flags=re.I).lower()


def discover(root):
    imgs, msks = {}, {}
    for dp, _, fns in os.walk(root):
        for fn in sorted(fns):
            if fn.lower().endswith(IMG_EXT):
                p = os.path.join(dp, fn)
                (msks if _is_mask_path(p) else imgs).setdefault(_key(p), p)
    pairs = []
    for k in sorted(set(imgs) & set(msks)):
        m = PAT_RE.search(os.path.basename(imgs[k]))
        if m:
            pairs.append((imgs[k], msks[k], ('TCGA-%s-%s' % (m.group(1), m.group(2))).upper(), m.group(1).upper()))
    return pairs, len(imgs), len(msks)


def read_mask(p):
    a = np.array(Image.open(p))
    a = a[..., 0] if a.ndim == 3 else a
    return np.minimum(a, 255).astype(np.uint8)          # codes > 255 (16-bit PNGs) fall into the ignore range


def tile_coords(h, w, P):
    if h < 1.5 * P and w < 1.5 * P:
        return [(0, 0, h, w)]
    return [(y, x, P, P) for y in range(0, h - P + 1, P) for x in range(0, w - P + 1, P)]


def make_splits(patients, sites, seed, n_test_sites=7, val_frac=0.15):
    pat_site = dict(zip(patients, sites))
    site_pats = defaultdict(list)
    for p, s in sorted(pat_site.items()):
        site_pats[s].append(p)
    all_sites = sorted(site_pats)
    npat = len(pat_site)
    rng = np.random.RandomState(seed)
    n_test_sites = max(1, min(n_test_sites, len(all_sites) // 3))
    best, best_err = None, 9
    for _ in range(2000):                                   # aim for ~27% of patients in test, as in BCSS
        ts = sorted(rng.choice(all_sites, n_test_sites, replace=False).tolist())
        err = abs(sum(len(site_pats[s]) for s in ts) / npat - 0.27)
        if err < best_err:
            best, best_err = ts, err
        if err < 0.03:
            break
    test_sites = best
    rest = sorted([s for s in all_sites if s not in test_sites], key=lambda s: (-len(site_pats[s]), s))
    n_rest = sum(len(site_pats[s]) for s in rest)
    cands = [s for s in rest[1:] if len(site_pats[s]) >= 2] or [s for s in rest if len(site_pats[s]) >= 2] or rest
    forget_site = min(cands, key=lambda s: (abs(len(site_pats[s]) - 0.07 * n_rest), s))
    cands2 = [s for s in rest if s != forget_site and len(site_pats[s]) >= 2] or [s for s in rest if s != forget_site]
    ref_site = min(cands2, key=lambda s: (abs(len(site_pats[s]) - 5), s))
    pool = sorted(p for s in rest if s not in (ref_site, forget_site) for p in site_pats[s])
    rng.shuffle(pool)
    n_val = max(1, int(round(val_frac * len(pool))))
    val_p = sorted(pool[:n_val])
    train_p = sorted(pool[n_val:] + site_pats[forget_site])
    return dict(test_sites=test_sites, ref_site=ref_site, forget_site=forget_site,
                test_patients=sorted(p for s in test_sites for p in site_pats[s]),
                ref_patients=sorted(site_pats[ref_site]), forget_site_patients=sorted(site_pats[forget_site]),
                val_patients=val_p, train_patients=train_p)


def stage_prepare(a):
    cdir = os.path.join(a.work, 'cache')
    os.makedirs(cdir, exist_ok=True)
    if os.path.exists(os.path.join(cdir, 'meta.json')) and not a.force:
        print('cache exists, skipping prepare (use --force to rebuild)')
        return
    pairs, ni, nm = discover(a.data_root)
    print(f'found {ni} images and {nm} masks under {a.data_root}; {len(pairs)} pairs carry a TCGA barcode')
    if not pairs:
        raise SystemExit('No image/mask pairs found. Masks must sit in a folder whose name contains "mask" '
                         '(or end in _mask) and share the image file name; names must contain a TCGA barcode.')
    if a.max_pairs:
        pairs = pairs[:a.max_pairs]
    vals = set()
    for _, mp, _, _ in pairs[:min(60, len(pairs))]:
        vals |= set(np.unique(read_mask(mp)).tolist())
    print('raw mask values in a sample of masks:', sorted(vals)[:40])
    if max(vals) <= 5:
        print('NOTE: mask values <= 5; assuming BCSS codes 1 tumour, 2 stroma, 3 inflammatory, 4 necrosis, 0 ignore.')
    lut = bcss_lut()
    P, S = a.patch, a.img
    plan = []                                              # first pass: masks only, decide which tiles to keep
    for ip, mp, pat, site in pairs:
        m = lut[read_mask(mp)]
        for (y, x, h, w) in tile_coords(m.shape[0], m.shape[1], P):
            mt = m[y:y + h, x:x + w]
            if (mt != IGN).mean() >= a.min_valid:
                plan.append((ip, mp, pat, site, y, x, h, w))
    n = len(plan)
    print(f'{n} tiles kept (patch {P}px, >= {a.min_valid:.0%} annotated), resized to {S}px')
    X = np.lib.format.open_memmap(os.path.join(cdir, 'X.npy'), 'w+', np.uint8, (n, S, S, 3))
    Y = np.lib.format.open_memmap(os.path.join(cdir, 'Y.npy'), 'w+', np.uint8, (n, S, S))
    cur_ip, img, msk = None, None, None
    for i, (ip, mp, pat, site, y, x, h, w) in enumerate(plan):
        if ip != cur_ip:
            img = np.array(Image.open(ip).convert('RGB')); msk = lut[read_mask(mp)]; cur_ip = ip
        X[i] = np.array(Image.fromarray(img[y:y + h, x:x + w]).resize((S, S), Image.BILINEAR))
        Y[i] = np.array(Image.fromarray(msk[y:y + h, x:x + w]).resize((S, S), Image.NEAREST))
    X.flush(); Y.flush()
    patients = [t[2] for t in plan]; sites = [t[3] for t in plan]
    sp = make_splits(sorted(set(patients)), [dict(zip(patients, sites))[p] for p in sorted(set(patients))], a.split_seed)
    meta = dict(patient=patients, site=sites, source=[os.path.basename(t[0]) for t in plan],
                tile=[t[4:] for t in plan], img=S, patch=P, splits=sp)
    jdump(meta, os.path.join(cdir, 'meta.json'))
    cnt = {k: len(v) for k, v in sp.items() if k.endswith('patients')}
    print('splits (patients):', cnt, '| test sites:', sp['test_sites'], '| S1-I forget site:', sp['forget_site'],
          '| reference site:', sp['ref_site'])


class Data:
    def __init__(self, work):
        c = os.path.join(work, 'cache')
        self.X = np.load(os.path.join(c, 'X.npy'), mmap_mode='r')
        self.Y = np.array(np.load(os.path.join(c, 'Y.npy'), mmap_mode='r'))
        self.meta = jload(os.path.join(c, 'meta.json'))
        self.sp = self.meta['splits']
        self.patient = np.array(self.meta['patient'])
        self.site = np.array(self.meta['site'])

    def idx_of(self, patients):
        return np.where(np.isin(self.patient, list(patients)))[0]

    def batch(self, idx, Y=None, aug=False):
        idx = np.sort(np.asarray(idx))
        x = torch.from_numpy(np.ascontiguousarray(self.X[idx])).to(DEV)
        x = x.permute(0, 3, 1, 2).float().div_(255.).sub_(0.5).div_(0.25)
        y = torch.from_numpy(np.ascontiguousarray((self.Y if Y is None else Y)[idx])).to(DEV).long()
        if aug:
            if random.random() < 0.5:
                x, y = x.flip(3), y.flip(2)
            if random.random() < 0.5:
                x, y = x.flip(2), y.flip(1)
            k = random.randint(0, 3)
            if k:
                x, y = torch.rot90(x, k, (2, 3)), torch.rot90(y, k, (1, 2))
        return x, y


# ----------------------------------------------------------------------------- model
class ConvBlock(nn.Sequential):
    def __init__(self, i, o):
        super().__init__(nn.Conv2d(i, o, 3, padding=1, bias=False), nn.BatchNorm2d(o), nn.ReLU(inplace=True),
                         nn.Conv2d(o, o, 3, padding=1, bias=False), nn.BatchNorm2d(o), nn.ReLU(inplace=True))


class AttGate(nn.Module):
    def __init__(self, g, x, inter):
        super().__init__()
        self.wg = nn.Sequential(nn.Conv2d(g, inter, 1, bias=False), nn.BatchNorm2d(inter))
        self.wx = nn.Sequential(nn.Conv2d(x, inter, 1, bias=False), nn.BatchNorm2d(inter))
        self.psi = nn.Sequential(nn.Conv2d(inter, 1, 1), nn.BatchNorm2d(1), nn.Sigmoid())

    def forward(self, g, x):
        return x * self.psi(F.relu(self.wg(g) + self.wx(x)))


class UpBlock(nn.Module):
    def __init__(self, i, skip, o, attention):
        super().__init__()
        self.up = nn.Sequential(nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False),
                                nn.Conv2d(i, o, 1, bias=False), nn.BatchNorm2d(o), nn.ReLU(inplace=True))
        self.att = AttGate(o, skip, max(o // 2, 1)) if attention else None
        self.conv = ConvBlock(o + skip, o)

    def forward(self, x, skip):
        x = self.up(x)
        if self.att is not None:
            skip = self.att(x, skip)
        return self.conv(torch.cat([x, skip], 1))


class SegNet(nn.Module):
    """Attention U-Net (attention=True) or U-Net (attention=False). Encoder = self.enc, decoder = self.dec + self.head."""

    def __init__(self, base=32, attention=True, n_cls=NC):
        super().__init__()
        c = [base * 2 ** i for i in range(5)]
        self.enc = nn.ModuleList([ConvBlock(3, c[0])] + [ConvBlock(c[i - 1], c[i]) for i in range(1, 5)])
        self.dec = nn.ModuleList([UpBlock(c[4], c[3], c[3], attention), UpBlock(c[3], c[2], c[2], attention),
                                  UpBlock(c[2], c[1], c[1], attention), UpBlock(c[1], c[0], c[0], attention)])
        self.head = nn.Conv2d(c[0], n_cls, 1)

    def forward(self, x):
        feats = []
        for i, e in enumerate(self.enc):
            x = e(x if i == 0 else F.max_pool2d(x, 2))
            feats.append(x)
        x = feats[-1]
        for d, skip in zip(self.dec, reversed(feats[:-1])):
            x = d(x, skip)
        return self.head(x)


class LoRAConv(nn.Module):
    """h = W*x + (alpha/r) B*(A*x); A: k x k conv c_in->r, B: 1x1 conv r->c_out, B initialised to zero."""

    def __init__(self, conv, r=8, alpha=16):
        super().__init__()
        self.base = conv
        for p in conv.parameters():
            p.requires_grad_(False)
        self.A = nn.Conv2d(conv.in_channels, r, conv.kernel_size, conv.stride, conv.padding, conv.dilation, bias=False)
        self.B = nn.Conv2d(r, conv.out_channels, 1, bias=False)
        nn.init.kaiming_uniform_(self.A.weight, a=math.sqrt(5))
        nn.init.zeros_(self.B.weight)
        self.scale = alpha / r

    def forward(self, x):
        return self.base(x) + self.scale * self.B(self.A(x))


def inject_lora(model, r=8, alpha=16, where=('dec', 'head')):
    for p in model.parameters():
        p.requires_grad_(False)

    def rec(mod):
        for name, ch in mod.named_children():
            if isinstance(ch, nn.Conv2d):
                setattr(mod, name, LoRAConv(ch, r, alpha).to(DEV))
            else:
                rec(ch)
    for w in where:
        sub = getattr(model, w)
        if isinstance(sub, nn.Conv2d):
            setattr(model, w, LoRAConv(sub, r, alpha).to(DEV))
        else:
            rec(sub)
    return [p for p in model.parameters() if p.requires_grad]


def bn_eval(model):
    for m in model.modules():
        if isinstance(m, nn.BatchNorm2d):
            m.eval()


# ----------------------------------------------------------------------------- losses
def soft_dice_loss(logits, y):
    valid = (y != IGN).unsqueeze(1).float()
    p = logits.float().softmax(1) * valid
    oh = F.one_hot(y.clamp(max=NC - 1), NC).permute(0, 3, 1, 2).float() * valid
    inter = (p * oh).sum((0, 2, 3)); den = p.sum((0, 2, 3)) + oh.sum((0, 2, 3))
    present = oh.sum((0, 2, 3)) > 0
    if not present.any():
        return logits.sum() * 0
    return 1 - ((2 * inter + 1) / (den + 1))[present].mean()


def seg_loss(logits, y):
    return F.cross_entropy(logits.float(), y, ignore_index=IGN) + soft_dice_loss(logits, y)


def pixel_kl(student, teacher, y, T=1.0, w=None):
    """mean over valid pixels of T^2 KL(softmax(t/T) || softmax(s/T)), optionally pixel-weighted."""
    ls = F.log_softmax(student.float() / T, 1)
    pt = F.softmax(teacher.float() / T, 1)
    kl = (pt * (pt.clamp_min(1e-8).log() - ls)).sum(1)
    wt = (y != IGN).float() if w is None else w * (y != IGN).float()
    return (wt * kl).sum() / wt.sum().clamp_min(1.0) * T * T


def boundary_weights(pred, beta=4.0, sigma=4.0):
    """w(p) = 1 + beta exp(-d(p)^2 / 2 sigma^2), d = chessboard distance to the nearest class boundary of pred."""
    oh = F.one_hot(pred, NC).permute(0, 3, 1, 2).float()
    mx = F.max_pool2d(oh, 3, 1, 1)
    mn = -F.max_pool2d(-oh, 3, 1, 1)
    b = ((mx - mn).sum(1) > 0).float()                       # B x H x W boundary map
    K = int(math.ceil(3 * sigma))
    d = torch.full_like(b, K + 1.0)
    d[b > 0] = 0
    cur = b.unsqueeze(1)
    for k in range(1, K + 1):
        cur = F.max_pool2d(cur, 3, 1, 1)
        d = torch.where((cur[:, 0] > 0) & (d > K), torch.full_like(d, float(k)), d)
    return 1.0 + beta * torch.exp(-d ** 2 / (2 * sigma ** 2))


# ----------------------------------------------------------------------------- training and evaluation
def new_model(a):
    return SegNet(base=a.base, attention=(a.arch == 'attunet')).to(DEV)


def train_model(a, data, idx, Y, seed, name):
    path = os.path.join(a.work, 'ckpt', name + '.pt')
    model = new_model(a)
    if os.path.exists(path):
        model.load_state_dict(torch.load(path, map_location=DEV))
        model.eval()
        return model, None
    os.makedirs(os.path.dirname(path), exist_ok=True)
    set_seed(seed)
    idx = np.asarray(idx)
    steps_per_ep = max(1, len(idx) // a.bs)
    total = a.epochs * steps_per_ep
    opt = torch.optim.AdamW(model.parameters(), lr=a.lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=a.lr, total_steps=total, pct_start=0.1)
    scaler = grad_scaler()
    rng = np.random.RandomState(seed)
    t0 = time.time()
    model.train()
    for ep in range(a.epochs):
        perm = rng.permutation(idx)
        run = 0.0
        for s in range(steps_per_ep):
            x, y = data.batch(perm[s * a.bs:(s + 1) * a.bs], Y, aug=True)
            with autocast():
                loss = seg_loss(model(x), y)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(opt); scaler.update(); sched.step()
            run += loss.item()
        if ep == 0 or (ep + 1) % max(1, a.epochs // 5) == 0 or ep == a.epochs - 1:
            print(f'  [{name}] epoch {ep + 1}/{a.epochs} loss {run / steps_per_ep:.4f} ({time.time() - t0:.0f}s)', flush=True)
    model.eval()
    torch.save(model.state_dict(), path)
    return model, time.time() - t0


def _bnd(m):
    return m & ~ndimage.binary_erosion(m, border_value=1)


def hd95_bf(gt, pr, tol):
    if not gt.any() and not pr.any():
        return None, None
    if not gt.any() or not pr.any():
        return float(math.hypot(*gt.shape)), 0.0
    bg, bp = _bnd(gt), _bnd(pr)
    d_pg = ndimage.distance_transform_edt(~bg)[bp]
    d_gp = ndimage.distance_transform_edt(~bp)[bg]
    hd = float(np.percentile(np.concatenate([d_pg, d_gp]), 95))
    pr_, rc_ = (d_pg <= tol).mean(), (d_gp <= tol).mean()
    return hd, float(2 * pr_ * rc_ / (pr_ + rc_)) if pr_ + rc_ > 0 else 0.0


@torch.no_grad()
def evaluate(model, data, idx, Y=None, boundary=True, tol=2, bs=32):
    """Dataset-level Dice/IoU per class, tumour HD95 and boundary F1 per patch, and per-patch MIA features."""
    model.eval()
    idx = np.sort(np.asarray(idx))
    tp, fp, fn = np.zeros(NC), np.zeros(NC), np.zeros(NC)
    hds, bfs, feats = [], [], []
    for s in range(0, len(idx), bs):
        x, y = data.batch(idx[s:s + bs], Y)
        with autocast():
            lg = model(x)
        lg = lg.float()
        valid = y != IGN
        ce = F.cross_entropy(lg, y, ignore_index=IGN, reduction='none')
        p = lg.softmax(1)
        ent = -(p * p.clamp_min(1e-8).log()).sum(1)
        mp = p.max(1).values
        nv = valid.sum((1, 2)).clamp_min(1).float()
        mce = (ce * valid).sum((1, 2)) / nv
        sce = (((ce - mce[:, None, None]) ** 2) * valid).sum((1, 2)).div(nv).sqrt()
        feats.append(torch.stack([mce, sce, (ent * valid).sum((1, 2)) / nv, (mp * valid).sum((1, 2)) / nv], 1).cpu().numpy())
        pred = lg.argmax(1)
        for c in range(NC):
            pc, yc = (pred == c) & valid, (y == c) & valid
            tp[c] += (pc & yc).sum().item(); fp[c] += (pc & ~yc).sum().item(); fn[c] += (~pc & yc).sum().item()
        if boundary:
            pn, yn, vn = pred.cpu().numpy(), y.cpu().numpy(), valid.cpu().numpy()
            for j in range(len(pn)):
                h, f = hd95_bf((yn[j] == TUM) & vn[j], (pn[j] == TUM) & vn[j], tol)
                if h is not None:
                    hds.append(h); bfs.append(f)
    with np.errstate(invalid='ignore', divide='ignore'):
        dice = np.where(2 * tp + fp + fn > 0, 2 * tp / (2 * tp + fp + fn), np.nan)
        iou = np.where(tp + fp + fn > 0, tp / (tp + fp + fn), np.nan)
    out = {'mdice': float(np.nanmean(dice)), 'miou': float(np.nanmean(iou)),
           'hd95_tum': float(np.mean(hds)) if hds else float('nan'),
           'bf_tum': float(np.mean(bfs)) if bfs else float('nan')}
    for c, n in enumerate(CLASSES):
        out['dice_' + n] = float(dice[c])
    return out, np.concatenate(feats) if feats else np.zeros((0, 4))


@torch.no_grad()
def mean_pixel_loss(model, data, idx, Y=None, bs=32):
    model.eval()
    tot, cnt = 0.0, 0
    for s in range(0, len(idx), bs):
        x, y = data.batch(np.asarray(idx)[s:s + bs], Y)
        with autocast():
            lg = model(x)
        ce = F.cross_entropy(lg.float(), y, ignore_index=IGN, reduction='sum')
        tot += ce.item(); cnt += (y != IGN).sum().item()
    return tot / max(cnt, 1)


@torch.no_grad()
def per_patch_loss(model, data, idx, Y=None, bs=32):
    model.eval()
    out = []
    for s in range(0, len(idx), bs):
        x, y = data.batch(np.asarray(idx)[s:s + bs], Y)
        with autocast():
            lg = model(x)
        ce = F.cross_entropy(lg.float(), y, ignore_index=IGN, reduction='none')
        v = (y != IGN).float()
        out.append(((ce * v).sum((1, 2)) / v.sum((1, 2)).clamp_min(1)).cpu().numpy())
    return np.concatenate(out)


def mia_efficacy(fR, fH, fF, seed=0):
    """Attack trained to separate retained (member) from unseen (non-member) patches, applied to the forget set.
    Returns the fraction of forget patches classified as non-members."""
    from sklearn.linear_model import LogisticRegression
    from sklearn.pipeline import make_pipeline
    from sklearn.preprocessing import StandardScaler
    rng = np.random.RandomState(seed)
    n = min(len(fR), len(fH), 400)
    if n < 4 or len(fF) == 0:
        return float('nan')
    X = np.vstack([fR[rng.choice(len(fR), n, replace=False)], fH[rng.choice(len(fH), n, replace=False)]])
    yy = np.r_[np.ones(n), np.zeros(n)]
    clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)).fit(X, yy)
    return float((clf.predict(fF) == 0).mean())


def forget_unseen_auc(lossF, lossH):
    """AUC for separating unseen (positive) from forget patches by per-patch loss; 0.5 = indistinguishable."""
    from sklearn.metrics import roc_auc_score
    if len(lossF) == 0 or len(lossH) == 0:
        return float('nan')
    return float(roc_auc_score(np.r_[np.zeros(len(lossF)), np.ones(len(lossH))], np.r_[lossF, lossH]))


# ----------------------------------------------------------------------------- unlearning methods
class Ctx:
    """Everything an unlearning method needs for one request."""

    def __init__(self, a, data, m0, F_idx, R_idx, H_idx, Y, scenario, seed, val_idx=None, Yval=None):
        self.a, self.data, self.m0 = a, data, m0
        self.F, self.R, self.H = np.asarray(F_idx), np.asarray(R_idx), np.asarray(H_idx)
        self.Y, self.scenario, self.seed = Y, scenario, seed
        self.val, self.Yval = (np.asarray(val_idx) if val_idx is not None else None), Yval
        self.rng = np.random.RandomState(seed)
        self.info = {}

    def fb(self, n=None):
        return self.data.batch(self.rng.choice(self.F, n or self.a.ubs, replace=len(self.F) < (n or self.a.ubs)), self.Y)

    def rb(self, n=None):
        return self.data.batch(self.rng.choice(self.R, n or self.a.ubs, replace=len(self.R) < (n or self.a.ubs)), self.Y)


def _clone(m0):
    m = copy.deepcopy(m0)
    m.train(); bn_eval(m)
    for p in m.parameters():
        p.requires_grad_(True)
    return m


def _teacher(ctx):
    ctx.m0.eval()
    for p in ctx.m0.parameters():
        p.requires_grad_(False)
    return ctx.m0


def m_finetune(ctx):
    m = _clone(ctx.m0)
    opt = torch.optim.Adam(m.parameters(), lr=ctx.a.ulr)
    for _ in range(ctx.a.usteps):
        x, y = ctx.rb()
        loss = seg_loss(m(x), y)
        opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
    return m


def m_neggrad_plus(ctx, beta=0.9):
    m = _clone(ctx.m0)
    opt = torch.optim.Adam(m.parameters(), lr=ctx.a.ulr)
    for _ in range(ctx.a.usteps):
        xr, yr = ctx.rb(); xf, yf = ctx.fb()
        loss = beta * seg_loss(m(xr), yr) - (1 - beta) * F.cross_entropy(m(xf).float(), yf, ignore_index=IGN)
        opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
    return m


def m_scrub(ctx):
    t = _teacher(ctx); m = _clone(ctx.m0)
    opt = torch.optim.Adam(m.parameters(), lr=ctx.a.ulr * 0.5)
    msteps = ctx.a.usteps // 2
    for it in range(ctx.a.usteps):
        if it < msteps:                                    # max-step: move away from the teacher on forget data
            xf, yf = ctx.fb()
            with torch.no_grad():
                tf = t(xf)
            loss = -pixel_kl(m(xf), tf, yf)
            opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
        xr, yr = ctx.rb()                                  # min-step: stay close to the teacher on retained data
        with torch.no_grad():
            tr = t(xr)
        s = m(xr)
        loss = pixel_kl(s, tr, yr) + F.cross_entropy(s.float(), yr, ignore_index=IGN)
        opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
    return m


def _fisher(m, data, idx, Y, bs=8):
    m.eval()
    imp = {n: torch.zeros_like(p) for n, p in m.named_parameters()}
    nb = 0
    for s in range(0, len(idx), bs):
        x, y = data.batch(np.asarray(idx)[s:s + bs], Y)
        m.zero_grad(set_to_none=True)
        F.cross_entropy(m(x).float(), y, ignore_index=IGN).backward()
        for n, p in m.named_parameters():
            if p.grad is not None:
                imp[n] += p.grad.detach() ** 2
        nb += 1
    return {n: v / max(nb, 1) for n, v in imp.items()}


def m_ssd(ctx, alpha=10.0, lam=1.0):
    m = _clone(ctx.m0)
    tr_all = np.r_[ctx.R, ctx.F]
    sub = ctx.rng.choice(tr_all, min(len(tr_all), 512), replace=False)
    I_D = _fisher(m, ctx.data, sub, ctx.Y)
    I_F = _fisher(m, ctx.data, ctx.F, ctx.Y)
    with torch.no_grad():
        for n, p in m.named_parameters():
            sel = I_F[n] > alpha * I_D[n]
            beta = (lam * I_D[n] / I_F[n].clamp_min(1e-12)).clamp(max=1.0)
            p[sel] = p[sel] * beta[sel]
    m.eval()
    return m


def m_salun(ctx, frac=0.5):
    m = _clone(ctx.m0)
    g = {n: torch.zeros_like(p) for n, p in m.named_parameters()}
    m.eval()
    for s in range(0, len(ctx.F), 8):
        x, y = ctx.data.batch(ctx.F[s:s + 8], ctx.Y)
        m.zero_grad(set_to_none=True)
        F.cross_entropy(m(x).float(), y, ignore_index=IGN).backward()
        for n, p in m.named_parameters():
            if p.grad is not None:
                g[n] += p.grad.detach().abs()
    flat = torch.cat([v.flatten() for v in g.values()])
    thr = flat.kthvalue(max(1, int((1 - frac) * flat.numel()))).values
    mask = {n: (v >= thr).float() for n, v in g.items()}
    m.train(); bn_eval(m)
    opt = torch.optim.Adam(m.parameters(), lr=ctx.a.ulr)
    for _ in range(ctx.a.usteps):
        xf, yf = ctx.fb()
        yr_ = yf.clone()                                   # random relabelling: each class -> a different random class
        for j in range(len(yf)):
            perm = torch.randperm(NC, device=DEV)
            while (perm == torch.arange(NC, device=DEV)).any():
                perm = torch.randperm(NC, device=DEV)
            v = yf[j] != IGN
            yr_[j][v] = perm[yf[j][v]]
        xr, yr = ctx.rb()
        loss = F.cross_entropy(m(xf).float(), yr_, ignore_index=IGN) + seg_loss(m(xr), yr)
        opt.zero_grad(set_to_none=True); loss.backward()
        for n, p in m.named_parameters():
            if p.grad is not None:
                p.grad.mul_(mask[n])
        opt.step()
    return m


def m_e2r(ctx):
    """Erase-to-Retain adapted to multi-class BCSS: decoder LoRA (r=8, alpha=32); forget patches pushed to the
    default class (stroma) with weight 3; retain: CE+Dice, KD (T=2) and an L2 guard (0.05) to the teacher."""
    t = _teacher(ctx); m = copy.deepcopy(ctx.m0)
    params = inject_lora(m, r=8, alpha=32); m.eval()
    opt = torch.optim.Adam(params, lr=ctx.a.alr)
    for _ in range(ctx.a.usteps):
        xf, yf = ctx.fb()
        tgt = torch.full_like(yf, STR); tgt[yf == IGN] = IGN
        lf = F.cross_entropy(m(xf).float(), tgt, ignore_index=IGN)
        xr, yr = ctx.rb()
        s = m(xr)
        with torch.no_grad():
            tr = t(xr)
        lr_ = seg_loss(s, yr) + pixel_kl(s, tr, yr, T=2.0) + 0.05 * F.mse_loss(s.float(), tr.float())
        loss = 3.0 * lf + lr_
        opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
    return m


@torch.no_grad()
def _val_score(m, ctx):
    r, _ = evaluate(m, ctx.data, ctx.val[:256], ctx.Yval)
    return (r['dice_tumour'] if not math.isnan(r['dice_tumour']) else 0) + (r['bf_tum'] if not math.isnan(r['bf_tum']) else 0)


def m_cabu(ctx, beta=4.0, calibrated=True, stopping=True, encoder=False, lambda_f=None):
    a, t = ctx.a, _teacher(ctx)
    lam_f = a.lambda_f if lambda_f is None else lambda_f
    m = copy.deepcopy(ctx.m0)
    params = inject_lora(m, r=a.rank, alpha=a.lora_alpha, where=('enc', 'dec', 'head') if encoder else ('dec', 'head'))
    m.eval()                                               # BN statistics frozen; adapters still receive gradients
    if ctx.scenario == 'S1':
        m_star = mean_pixel_loss(t, ctx.data, ctx.H, ctx.Y)
    else:
        m_star = math.log(NC)
    opt = torch.optim.Adam(params, lr=a.alr)
    T_max = a.steps_s1 if ctx.scenario == 'S1' else a.steps_s2
    traj = {'m_star': m_star, 'step': [], 'm_F': [], 'm_star_t': [], 'ks_p': [], 'gap': [], 'val': []}
    best, best_state, bad, stop_step = -1e9, None, 0, T_max
    subF = ctx.rng.choice(ctx.F, min(len(ctx.F), 128), replace=False)
    subH = ctx.rng.choice(ctx.H, min(len(ctx.H), 128), replace=False)
    for it in range(1, T_max + 1):
        xf, yf = ctx.fb()
        mF = F.cross_entropy(m(xf).float(), yf, ignore_index=IGN)
        LF = (mF - m_star) ** 2 if calibrated else -torch.clamp(mF, max=math.log(NC))
        xr, yr = ctx.rb()
        s = m(xr)
        with torch.no_grad():
            tr = t(xr)
            w = boundary_weights(tr.argmax(1), beta, a.sigma) if beta > 0 else None
        LR = seg_loss(s, yr) + a.lambda_kd * pixel_kl(s, tr, yr, T=2.0, w=w)
        loss = lam_f * LF + LR
        opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
        if it % a.check_every == 0 or it == T_max:
            traj['step'].append(it); traj['m_F'].append(float(mF.item())); traj['m_star_t'].append(float(m_star))
            if ctx.scenario == 'S1':
                # the unseen level is re-estimated for the model as it now stands (the retain term keeps changing it)
                m_star = mean_pixel_loss(m, ctx.data, subH, ctx.Y)
                lF = per_patch_loss(m, ctx.data, subF, ctx.Y); lH = per_patch_loss(m, ctx.data, subH, ctx.Y)
                p = float(ks_2samp(lF, lH).pvalue); gap = float(abs(lF.mean() - lH.mean()) / max(lH.mean(), 1e-8))
                traj['ks_p'].append(p); traj['gap'].append(gap)
                if stopping and p >= 0.05 and gap <= 0.05:
                    stop_step = it
                    break
            else:
                sc = _val_score(m, ctx)
                traj['val'].append(sc)
                if sc > best:
                    best, bad = sc, 0
                    best_state = {k: v.detach().clone() for k, v in m.state_dict().items()}
                else:
                    bad += 1
                    if stopping and bad >= 3:
                        stop_step = it
                        break
    if ctx.scenario == 'S2' and best_state is not None:
        m.load_state_dict(best_state)
    traj['stop_step'] = stop_step
    ctx.info['traj'] = traj
    m.eval()
    return m


METHODS = {
    'finetune': m_finetune, 'neggrad_plus': m_neggrad_plus, 'scrub': m_scrub, 'ssd': m_ssd, 'salun': m_salun,
    'e2r': m_e2r, 'cabu': m_cabu,
    'cabu_noboundary': lambda c: m_cabu(c, beta=0.0),
    'cabu_nocal': lambda c: m_cabu(c, calibrated=False),
    'cabu_nostop': lambda c: m_cabu(c, stopping=False),
    'cabu_encoder': lambda c: m_cabu(c, encoder=True),
    'cabu_lf0.5': lambda c: m_cabu(c, lambda_f=0.5),
    'cabu_lf2': lambda c: m_cabu(c, lambda_f=2.0),
    'cabu_lf5': lambda c: m_cabu(c, lambda_f=5.0),
}
LABELS = {'original': 'Original', 'retrain': 'Retrain', 'retrain_wo': 'Retrain w/o identified', 'oracle': 'Oracle (clean labels)',
          'finetune': 'Fine-tune', 'neggrad_plus': 'NegGrad+', 'scrub': 'SCRUB', 'ssd': 'SSD', 'salun': 'SalUn',
          'e2r': 'E2R (adapted)', 'cabu': 'CaBU (ours)', 'cabu_noboundary': 'CaBU w/o boundary weights',
          'cabu_nocal': 'CaBU w/o calibrated target', 'cabu_nostop': 'CaBU w/o stopping rule',
          'cabu_encoder': 'CaBU + encoder adapters', 'cabu_lf0.5': 'CaBU, λ_F = 0.5', 'cabu_lf2': 'CaBU, λ_F = 2',
          'cabu_lf5': 'CaBU, λ_F = 5'}


def n_trainable(m):
    lora = sum(p.numel() for mod in m.modules() if isinstance(mod, LoRAConv) for p in (mod.A.weight, mod.B.weight))
    return lora if lora else sum(p.numel() for p in m.parameters())


# ----------------------------------------------------------------------------- results bookkeeping
FIELDS = ['scenario', 'setting', 'op', 'rho', 'seed', 'method', 'time_s', 'trainable', 'stop_step', 'n_forget', 'n_retain',
          'test_mdice', 'test_miou', 'test_dice_tumour', 'test_dice_stroma', 'test_dice_inflammatory', 'test_dice_necrosis',
          'test_dice_other', 'test_hd95_tum', 'test_bf_tum', 'forget_mdice', 'forget_dice_tumour', 'forget_bf_tum',
          'retain_mdice', 'ref_mdice', 'mia_eff', 'fu_auc']


def results_path(a):
    return os.path.join(a.work, 'results.csv')


def done_keys(a):
    p = results_path(a)
    if not os.path.exists(p):
        return set()
    with open(p) as f:
        return {(r['scenario'], r['setting'], r['op'], r['rho'], r['seed'], r['method']) for r in csv.DictReader(f)}


def write_row(a, row):
    p = results_path(a)
    new = not os.path.exists(p)
    with open(p, 'a', newline='') as f:
        w = csv.DictWriter(f, fieldnames=FIELDS, extrasaction='ignore')
        if new:
            w.writeheader()
        w.writerow({k: row.get(k, '') for k in FIELDS})


def full_eval(m, ctx, test_idx, Ytest, F_labels):
    d = ctx.data
    rt, _ = evaluate(m, d, test_idx, Ytest)
    rf, fF = evaluate(m, d, ctx.F, F_labels)
    subR = ctx.rng.choice(ctx.R, min(len(ctx.R), 512), replace=False)
    rr, fR = evaluate(m, d, subR, ctx.Y, boundary=False)
    rh, fH = evaluate(m, d, ctx.H, ctx.Y, boundary=False)
    row = {'test_' + k: v for k, v in rt.items()}
    row.update(forget_mdice=rf['mdice'], forget_dice_tumour=rf['dice_tumour'], forget_bf_tum=rf['bf_tum'],
               retain_mdice=rr['mdice'], ref_mdice=rh['mdice'])
    if ctx.scenario == 'S1':
        row['mia_eff'] = mia_efficacy(fR, fH, fF, ctx.seed)
        row['fu_auc'] = forget_unseen_auc(fF[:, 0], fH[:, 0])
    return row


# ----------------------------------------------------------------------------- figures
def save_qual(path, data, idx, label_sets, models, titles):
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    from matplotlib.colors import ListedColormap
    cmap = ListedColormap(['#d62728', '#f2c6de', '#1f77b4', '#7f7f7f', '#2ca02c', '#ffffff'])
    idx = np.asarray(idx)
    preds = []
    with torch.no_grad():
        for m in models:
            x, _ = data.batch(idx)
            preds.append(m(x).argmax(1).cpu().numpy())
    order = np.argsort(idx)
    cols = 1 + len(label_sets) + len(models)
    fig, ax = plt.subplots(len(idx), cols, figsize=(2.1 * cols, 2.1 * len(idx)), squeeze=False)
    for r, i in enumerate(np.sort(idx)):
        ax[r, 0].imshow(data.X[i]); ax[r, 0].set_title('H&E' if r == 0 else '')
        for c, (nm, Yl) in enumerate(label_sets):
            lab = Yl[i].astype(int); lab[lab == IGN] = 5
            ax[r, 1 + c].imshow(lab, cmap=cmap, vmin=0, vmax=5, interpolation='nearest')
            if r == 0:
                ax[r, 1 + c].set_title(nm)
        for c, p in enumerate(preds):
            ax[r, 1 + len(label_sets) + c].imshow(p[r], cmap=cmap, vmin=0, vmax=5, interpolation='nearest')
            if r == 0:
                ax[r, 1 + len(label_sets) + c].set_title(titles[c])
    for x in ax.ravel():
        x.axis('off')
    from matplotlib.patches import Patch
    names = CLASSES + ['ignored']
    fig.legend(handles=[Patch(facecolor=cmap(k), edgecolor='k', label=names[k]) for k in range(6)],
               loc='lower center', ncol=6, fontsize=8, frameon=False)
    fig.tight_layout(rect=(0, 0.05, 1, 1)); fig.savefig(path, dpi=130); plt.close(fig)


# ----------------------------------------------------------------------------- stage S1
def run_methods(a, ctx, base_row, test_idx, Ytest, F_labels, methods, done):
    for meth in methods:
        key = (base_row['scenario'], base_row['setting'], str(base_row['op']), str(base_row['rho']), str(base_row['seed']), meth)
        if key in done:
            continue
        set_seed(ctx.seed)
        ctx.info = {}
        t0 = time.time()
        m = METHODS[meth](ctx)
        if CUDA:
            torch.cuda.synchronize()
        dt = time.time() - t0
        row = dict(base_row, method=meth, time_s=dt, trainable=n_trainable(m),
                   stop_step=ctx.info.get('traj', {}).get('stop_step', ''))
        row.update(full_eval(m, ctx, test_idx, Ytest, F_labels))
        write_row(a, row)
        if 'traj' in ctx.info:
            tdir = os.path.join(a.work, 'traj'); os.makedirs(tdir, exist_ok=True)
            jdump(ctx.info['traj'], os.path.join(tdir, '_'.join(map(str, key)) + '.json'))
        print(f"    {meth:16s} test mDice {row['test_mdice']:.3f}  tumour {row['test_dice_tumour']:.3f}  "
              f"forget mDice {row['forget_mdice']:.3f}  MIA {row.get('mia_eff', float('nan')):.3f}  "
              f"AUC {row.get('fu_auc', float('nan')):.3f}  {dt:.0f}s", flush=True)
        del m
        if CUDA:
            torch.cuda.empty_cache()


def stage_s1(a):
    data = Data(a.work); sp = data.sp
    tr_idx = data.idx_of(sp['train_patients']); val_idx = data.idx_of(sp['val_patients'])
    test_idx = data.idx_of(sp['test_patients']); ref_idx = data.idx_of(sp['ref_patients'])
    print(f'S1: train {len(tr_idx)} | val {len(val_idx)} | test {len(test_idx)} | reference site {len(ref_idx)} patches')
    m0, _ = train_model(a, data, tr_idx, data.Y, 0, 'orig_clean')
    done = done_keys(a)
    settings = []
    tp = np.array(sp['train_patients'])
    for frac in a.s1_fracs:
        for seed in a.seeds:
            rng = np.random.RandomState(1000 + seed)
            k = max(1, int(round(frac * len(tp))))
            settings.append((f'S1-P{int(round(frac * 100))}', seed, sorted(rng.choice(tp, k, replace=False).tolist()), val_idx))
    settings.append(('S1-I', 0, sp['forget_site_patients'], ref_idx))
    methods = a.methods_s1
    for setting, seed, fpats, H_idx in settings:
        F_idx = np.intersect1d(data.idx_of(fpats), tr_idx)
        R_idx = np.setdiff1d(tr_idx, F_idx)
        print(f'\n== {setting} seed {seed}: forget {len(fpats)} patients / {len(F_idx)} patches; retain {len(R_idx)}')
        ctx = Ctx(a, data, m0, F_idx, R_idx, H_idx, data.Y, 'S1', seed)
        base = dict(scenario='S1', setting=setting, op='', rho='', seed=seed, n_forget=len(F_idx), n_retain=len(R_idx))
        mr, tr_time = train_model(a, data, R_idx, data.Y, 0, f'retrain_{setting}_s{seed}')
        for name, mod, tt in (('original', m0, 0.0), ('retrain', mr, tr_time)):
            key = ('S1', setting, '', '', str(seed), name)
            if key not in done:
                row = dict(base, method=name, time_s=tt if tt is not None else '', trainable=n_trainable(mod))
                row.update(full_eval(mod, ctx, test_idx, data.Y, data.Y))
                write_row(a, row)
                print(f"    {name:16s} test mDice {row['test_mdice']:.3f}  forget mDice {row['forget_mdice']:.3f}  "
                      f"MIA {row['mia_eff']:.3f}  AUC {row['fu_auc']:.3f}")
        run_methods(a, ctx, base, test_idx, data.Y, data.Y, methods, done)
        qpath = os.path.join(a.work, 'fig_qual_S1.png')
        if setting == f'S1-P{int(round(max(a.s1_fracs) * 100))}' and seed == a.seeds[0] and not os.path.exists(qpath):
            ctx.info = {}
            mc = m_cabu(ctx)
            tum = np.array([(data.Y[i] == TUM).mean() for i in F_idx])
            pick = F_idx[np.argsort(-tum)[:3]]
            save_qual(qpath, data, pick, [('Ground truth', data.Y)], [m0, mr, mc], ['Original', 'Retrain', 'CaBU'])


# ----------------------------------------------------------------------------- stage S2
def corrupt(data, patients, op, frac, seed, r_native=40):   # 40 px at 0.25 um/px = 10 um margin error
    rng = np.random.RandomState(2000 + seed)
    k = max(1, int(round(frac * len(patients))))
    cp = sorted(rng.choice(patients, k, replace=False).tolist())
    idx = data.idx_of(cp)
    Yc = data.Y.copy()
    r = max(1, int(round(r_native * data.meta['img'] / data.meta['patch'])))
    disk = (np.add.outer(np.arange(-r, r + 1) ** 2, np.arange(-r, r + 1) ** 2) <= r * r)
    changed = []
    for i in idx:
        y = Yc[i]
        tum = y == TUM
        if not tum.any():
            continue
        if op == 'missed':
            lab, n = ndimage.label(tum)
            drop = rng.choice(np.arange(1, n + 1), max(1, n // 2), replace=False)
            y[np.isin(lab, drop)] = STR
        elif op == 'sloppy':
            dil = ndimage.binary_dilation(tum, structure=disk)
            y[dil & (y != IGN)] = TUM
        elif op == 'confusion':
            y[tum] = NEC
        else:
            raise ValueError(op)
        changed.append(i)
    return Yc, np.array(changed, dtype=int)


def stage_s2(a):
    data = Data(a.work); sp = data.sp
    tr_idx = data.idx_of(sp['train_patients']); val_idx = data.idx_of(sp['val_patients'])
    test_idx = data.idx_of(sp['test_patients'])
    oracle, _ = train_model(a, data, tr_idx, data.Y, 0, 'orig_clean')
    done = done_keys(a)
    methods = a.methods_s2
    for op in a.ops:
        Yc, corr = corrupt(data, sp['train_patients'], op, 0.20, 0)
        print(f'\n== S2 {op}: {len(corr)} corrupted patches in {len(tr_idx)} training patches')
        m0, _ = train_model(a, data, tr_idx, Yc, 0, f'orig_{op}')
        for rho in a.rhos:
            rng = np.random.RandomState(3000 + int(rho * 100))
            F_idx = np.sort(rng.choice(corr, max(1, int(math.ceil(rho * len(corr)))), replace=False))
            R_idx = np.setdiff1d(tr_idx, F_idx)
            ctx = Ctx(a, data, m0, F_idx, R_idx, val_idx, Yc, 'S2', 0, val_idx=val_idx, Yval=data.Y)
            base = dict(scenario='S2', setting=f'S2-{op}', op=op, rho=rho, seed=0, n_forget=len(F_idx), n_retain=len(R_idx))
            print(f'  rho={rho}: identified {len(F_idx)} of {len(corr)}')
            mrw, tt = train_model(a, data, R_idx, Yc, 0, f'retrainwo_{op}_{rho}')
            for name, mod, t_ in (('original', m0, 0.0), ('oracle', oracle, ''), ('retrain_wo', mrw, tt)):
                key = ('S2', f'S2-{op}', op, str(rho), '0', name)
                if key not in done:
                    row = dict(base, method=name, time_s=t_ if t_ is not None else '', trainable=n_trainable(mod))
                    row.update(full_eval(mod, ctx, test_idx, data.Y, data.Y))
                    write_row(a, row)
                    print(f"    {name:16s} test tumour Dice {row['test_dice_tumour']:.3f}  BF {row['test_bf_tum']:.3f}")
            # forget-set metrics are computed against the CLEAN labels: did the model stop reproducing the error?
            run_methods(a, ctx, base, test_idx, data.Y, data.Y, methods, done)
            qpath = os.path.join(a.work, f'fig_qual_S2_{op}.png')
            if abs(rho - sorted(a.rhos)[len(a.rhos) // 2]) < 1e-9 and not os.path.exists(qpath):
                ctx.info = {}
                mc = m_cabu(ctx)
                tum = np.array([(data.Y[i] == TUM).mean() for i in F_idx])
                pick = F_idx[np.argsort(-tum)[:3]]
                save_qual(qpath, data, pick, [('Clean label', data.Y), ('Corrupted label', Yc)], [m0, mc, oracle],
                          ['Original (corrupted)', 'CaBU', 'Oracle'])


# ----------------------------------------------------------------------------- report
def _f(x):
    try:
        return float(x)
    except (TypeError, ValueError):
        return float('nan')


def stage_report(a):
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    p = results_path(a)
    if not os.path.exists(p):
        raise SystemExit('no results.csv yet')
    with open(p) as f:
        rows = list(csv.DictReader(f))
    out = os.path.join(a.work, 'report'); os.makedirs(out, exist_ok=True)
    md = ['# CaBU results\n', f'Generated {time.strftime("%Y-%m-%d %H:%M")} from {len(rows)} result rows.\n']

    def ms(v):
        v = [x for x in v if not math.isnan(x)]
        if not v:
            return 'n/a'
        return f'{np.mean(v):.3f}' if len(v) == 1 else f'{np.mean(v):.3f} ± {np.std(v):.3f}'

    # ---- S1 tables
    s1 = [r for r in rows if r['scenario'] == 'S1']
    order = ['original', 'retrain', 'finetune', 'neggrad_plus', 'scrub', 'ssd', 'salun', 'e2r', 'cabu',
             'cabu_noboundary', 'cabu_nocal', 'cabu_nostop', 'cabu_encoder', 'cabu_lf0.5', 'cabu_lf2', 'cabu_lf5']
    agg_rows = []
    for setting in sorted({r['setting'] for r in s1}):
        rs = [r for r in s1 if r['setting'] == setting]
        ret = {r['seed']: r for r in rs if r['method'] == 'retrain'}
        md.append(f'\n## {setting}\n')
        md.append('| Method | Test mDice | Test tumour Dice | Tumour BF | Tumour HD95 (px) | Forget mDice | Gap to retrain (forget mDice) | MIA-efficacy | Forget-vs-unseen AUC | Avg gap | Time (s) | Trainable params |')
        md.append('|' + ' --- |' * 12)
        for meth in [m for m in order if any(r['method'] == m for r in rs)]:
            mr_ = [r for r in rs if r['method'] == meth]
            gaps, avg = [], []
            for r in mr_:
                rt = ret.get(r['seed'])
                if rt:
                    g1 = abs(_f(r['forget_mdice']) - _f(rt['forget_mdice']))
                    g2 = abs(_f(r['test_mdice']) - _f(rt['test_mdice']))
                    g3 = abs(_f(r['mia_eff']) - _f(rt['mia_eff']))
                    gaps.append(g1); avg.append(np.nanmean([g1, g2, g3]))
            vals = dict(test_mdice=ms([_f(r['test_mdice']) for r in mr_]), tumour=ms([_f(r['test_dice_tumour']) for r in mr_]),
                        bf=ms([_f(r['test_bf_tum']) for r in mr_]), hd=ms([_f(r['test_hd95_tum']) for r in mr_]),
                        fm=ms([_f(r['forget_mdice']) for r in mr_]), gap=ms(gaps), mia=ms([_f(r['mia_eff']) for r in mr_]),
                        auc=ms([_f(r['fu_auc']) for r in mr_]), avg=ms(avg), t=ms([_f(r['time_s']) for r in mr_]),
                        n=mr_[0]['trainable'])
            md.append(f"| {LABELS.get(meth, meth)} | {vals['test_mdice']} | {vals['tumour']} | {vals['bf']} | {vals['hd']} | {vals['fm']} | "
                      f"{vals['gap']} | {vals['mia']} | {vals['auc']} | {vals['avg']} | {vals['t']} | {vals['n']} |")
            agg_rows.append(dict(setting=setting, method=meth, **vals))
    with open(os.path.join(out, 'table_S1.csv'), 'w', newline='') as f:
        if agg_rows:
            w = csv.DictWriter(f, fieldnames=list(agg_rows[0].keys())); w.writeheader(); w.writerows(agg_rows)

    # ---- S1 trade-off figure
    if s1:
        settings = sorted({r['setting'] for r in s1})
        fig, axs = plt.subplots(1, len(settings), figsize=(4.6 * len(settings), 4), squeeze=False)
        for ax, setting in zip(axs[0], settings):
            rs = [r for r in s1 if r['setting'] == setting]
            ret = {r['seed']: r for r in rs if r['method'] == 'retrain'}
            for meth in [m for m in order if m not in ('retrain',) and any(r['method'] == m for r in rs)]:
                xs = [abs(_f(r['forget_mdice']) - _f(ret[r['seed']]['forget_mdice'])) for r in rs if r['method'] == meth and r['seed'] in ret]
                ys = [_f(r['test_mdice']) for r in rs if r['method'] == meth and r['seed'] in ret]
                if xs:
                    ax.scatter(np.mean(xs), np.mean(ys), s=60 if meth == 'cabu' else 30,
                               marker='*' if meth == 'cabu' else 'o', label=LABELS.get(meth, meth))
            rt = [_f(r['test_mdice']) for r in ret.values()]
            if rt:
                ax.axhline(np.mean(rt), ls='--', lw=1, c='gray')
            ax.set_xlabel('|forget mDice − retrain forget mDice|  (lower is better)')
            ax.set_ylabel('Test mDice'); ax.set_title(setting)
        axs[0][-1].legend(fontsize=7, loc='best')
        fig.tight_layout(); fig.savefig(os.path.join(out, 'fig_S1_tradeoff.png'), dpi=150); plt.close(fig)

    # ---- S2 tables and curves
    s2 = [r for r in rows if r['scenario'] == 'S2']
    order2 = ['original', 'oracle', 'retrain_wo', 'finetune', 'neggrad_plus', 'scrub', 'ssd', 'salun', 'e2r', 'cabu',
              'cabu_noboundary', 'cabu_nocal', 'cabu_nostop']
    for op in sorted({r['op'] for r in s2}):
        rs = [r for r in s2 if r['op'] == op]
        rhos = sorted({_f(r['rho']) for r in rs})
        for metric, title in (('test_dice_tumour', 'test tumour Dice'), ('test_bf_tum', 'test tumour boundary F1'),
                              ('test_hd95_tum', 'test tumour HD95 (px)'), ('forget_dice_tumour', 'tumour Dice on identified patches vs clean labels')):
            md.append(f'\n## S2 {op}: {title}\n')
            md.append('| Method | ' + ' | '.join(f'ρ = {x:g}' for x in rhos) + ' |')
            md.append('|' + ' --- |' * (len(rhos) + 1))
            for meth in [m for m in order2 if any(r['method'] == m for r in rs)]:
                cells = []
                for rho in rhos:
                    v = [_f(r[metric]) for r in rs if r['method'] == meth and _f(r['rho']) == rho]
                    cells.append(ms(v))
                md.append(f'| {LABELS.get(meth, meth)} | ' + ' | '.join(cells) + ' |')
        fig, ax = plt.subplots(figsize=(5.2, 4))
        for meth in [m for m in order2 if m not in ('cabu_noboundary', 'cabu_nocal', 'cabu_nostop') and any(r['method'] == m for r in rs)]:
            ys = [np.nanmean([_f(r['test_dice_tumour']) for r in rs if r['method'] == meth and _f(r['rho']) == rho] or [np.nan]) for rho in rhos]
            ax.plot(rhos, ys, marker='*' if meth == 'cabu' else 'o', lw=2 if meth == 'cabu' else 1,
                    ls='--' if meth in ('original', 'oracle') else '-', label=LABELS.get(meth, meth))
        ax.set_xlabel('Fraction of corrupted patches identified (ρ)'); ax.set_ylabel('Test tumour Dice')
        ax.set_title(f'S2 {op}'); ax.legend(fontsize=7)
        fig.tight_layout(); fig.savefig(os.path.join(out, f'fig_S2_{op}.png'), dpi=150); plt.close(fig)

    # ---- CaBU trajectory figure (S1)
    tdir = os.path.join(a.work, 'traj')
    if os.path.isdir(tdir):
        fs = sorted(f for f in os.listdir(tdir) if f.startswith('S1_') and f.endswith('_cabu.json'))
        if fs:
            fig, ax = plt.subplots(figsize=(5.2, 3.6))
            for fn in fs[:4]:
                t = jload(os.path.join(tdir, fn))
                l = ax.plot(t['step'], t['m_F'], marker='o', ms=3, label=fn.split('_')[1] + ' s' + fn.split('_')[4])
                ax.plot(t['step'], t.get('m_star_t') or [t['m_star']] * len(t['step']), ls='--', lw=1, c=l[0].get_color())
            ax.set_xlabel('Unlearning step'); ax.set_ylabel('Forget-batch loss m_F (dashed: target m*)')
            ax.legend(fontsize=7); fig.tight_layout(); fig.savefig(os.path.join(out, 'fig_cabu_trajectory.png'), dpi=150); plt.close(fig)

    for fn in os.listdir(a.work):
        if fn.startswith('fig_qual') and fn.endswith('.png'):
            shutil.copy(os.path.join(a.work, fn), os.path.join(out, fn))
    with open(os.path.join(out, 'tables.md'), 'w') as f:
        f.write('\n'.join(md) + '\n')
    sp = jload(os.path.join(a.work, 'cache', 'meta.json'))['splits']
    jdump(sp, os.path.join(out, 'splits.json'))
    print('\n'.join(md))
    print(f'\nreport written to {out}')


# ----------------------------------------------------------------------------- smoke test on synthetic data
def make_synthetic(root, n_sites=14, per_site=3, size=640, seed=0):
    rng = np.random.RandomState(seed)
    di, dm = os.path.join(root, 'images'), os.path.join(root, 'masks')
    os.makedirs(di, exist_ok=True); os.makedirs(dm, exist_ok=True)
    colours = {1: (120, 40, 140), 2: (230, 150, 200), 3: (60, 40, 160), 4: (240, 190, 210), 9: (250, 250, 250)}
    yy, xx = np.mgrid[:size, :size]
    for s in range(n_sites):
        site = 'A%d' % s if s < 10 else 'B%d' % (s - 10)
        for k in range(per_site):
            pid = 'TCGA-%s-%04d' % (site, 100 + k)
            lab = np.full((size, size), 2, np.uint8)
            for code, nblob, rad in ((1, 3, 90), (3, 4, 35), (4, 1, 60), (9, 2, 40)):
                for _ in range(nblob):
                    cy, cx = rng.randint(0, size, 2); ry, rx = rng.randint(rad // 2, rad * 2, 2)
                    lab[((yy - cy) / ry) ** 2 + ((xx - cx) / rx) ** 2 <= 1] = code
            if rng.rand() < 0.4:
                lab[:, :rng.randint(20, 80)] = 0
            img = np.zeros((size, size, 3), np.float32)
            for code, c in colours.items():
                img[lab == code] = c
            img = np.clip(img + rng.normal(0, 18, img.shape) + (s - 7) * 2, 0, 255).astype(np.uint8)
            name = f'{pid}-DX1_xmin{rng.randint(1e5)}_ymin{rng.randint(1e5)}_MPP-0.2500.png'
            Image.fromarray(img).save(os.path.join(di, name)); Image.fromarray(lab).save(os.path.join(dm, name))


def stage_smoke(a):
    root = os.path.join(a.work, 'synthetic_bcss')
    if not os.path.isdir(root):
        make_synthetic(root)
    a.data_root = root
    a.patch, a.img, a.epochs, a.base, a.bs = 256, 64, 2, 8, 8
    a.usteps, a.steps_s1, a.steps_s2, a.check_every, a.ubs = 30, 60, 60, 10, 4
    a.seeds, a.s1_fracs, a.ops, a.rhos = [0], [0.10], ['missed', 'sloppy'], [0.25, 0.5]
    a.sigma = 2.0
    for st in (stage_prepare, stage_s1, stage_s2, stage_report):
        print(f'\n######## {st.__name__}')
        st(a)


# ----------------------------------------------------------------------------- CLI
def main():
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument('stage', choices=['prepare', 's1', 's2', 'report', 'smoke', 'all'])
    ap.add_argument('--data_root', default='/kaggle/input')
    ap.add_argument('--work', default='/kaggle/working/cabu_run')
    ap.add_argument('--resume_dir', default='', help='copy cache/ckpt/results from an earlier run before starting')
    ap.add_argument('--force', action='store_true')
    ap.add_argument('--max_pairs', type=int, default=0)
    ap.add_argument('--patch', type=int, default=512)
    ap.add_argument('--img', type=int, default=256)
    ap.add_argument('--min_valid', type=float, default=0.5)
    ap.add_argument('--split_seed', type=int, default=0)
    ap.add_argument('--arch', default='attunet', choices=['attunet', 'unet'])
    ap.add_argument('--base', type=int, default=32)
    ap.add_argument('--epochs', type=int, default=30)
    ap.add_argument('--bs', type=int, default=16)
    ap.add_argument('--lr', type=float, default=1e-3)
    ap.add_argument('--ubs', type=int, default=8, help='forget and retain batch size during unlearning')
    ap.add_argument('--usteps', type=int, default=400, help='steps for baseline unlearning methods')
    ap.add_argument('--ulr', type=float, default=1e-4, help='learning rate for full-parameter baselines')
    ap.add_argument('--alr', type=float, default=1e-3, help='learning rate for adapter-based methods')
    ap.add_argument('--rank', type=int, default=8)
    ap.add_argument('--lora_alpha', type=float, default=16)
    ap.add_argument('--steps_s1', type=int, default=400)
    ap.add_argument('--steps_s2', type=int, default=600)
    ap.add_argument('--check_every', type=int, default=25)
    ap.add_argument('--lambda_f', type=float, default=1.0)
    ap.add_argument('--lambda_kd', type=float, default=1.0)
    ap.add_argument('--sigma', type=float, default=4.0)
    ap.add_argument('--seeds', type=int, nargs='+', default=[0, 1, 2])
    ap.add_argument('--s1_fracs', type=float, nargs='+', default=[0.05, 0.10])
    ap.add_argument('--ops', nargs='+', default=['missed', 'sloppy', 'confusion'])
    ap.add_argument('--rhos', type=float, nargs='+', default=[0.10, 0.25, 0.50, 1.00])
    ap.add_argument('--methods_s1', nargs='+', default=['finetune', 'neggrad_plus', 'scrub', 'ssd', 'salun', 'e2r', 'cabu',
                                                        'cabu_noboundary', 'cabu_nocal', 'cabu_nostop', 'cabu_encoder',
                                                        'cabu_lf0.5', 'cabu_lf2', 'cabu_lf5'])
    ap.add_argument('--methods_s2', nargs='+', default=['finetune', 'neggrad_plus', 'scrub', 'ssd', 'salun', 'e2r', 'cabu',
                                                        'cabu_noboundary', 'cabu_nocal'])
    a = ap.parse_args()
    os.makedirs(a.work, exist_ok=True)
    if a.resume_dir and os.path.isdir(a.resume_dir):
        for sub in ('cache', 'ckpt', 'traj'):
            s = os.path.join(a.resume_dir, sub)
            if os.path.isdir(s) and not os.path.isdir(os.path.join(a.work, sub)):
                shutil.copytree(s, os.path.join(a.work, sub))
        for fn in os.listdir(a.resume_dir):
            if (fn == 'results.csv' or fn.startswith('fig_qual')) and not os.path.exists(os.path.join(a.work, fn)):
                shutil.copy(os.path.join(a.resume_dir, fn), os.path.join(a.work, fn))
        print('resumed from', a.resume_dir)
    print(f'device: {DEV}; torch {torch.__version__}')
    stages = {'prepare': [stage_prepare], 's1': [stage_s1], 's2': [stage_s2], 'report': [stage_report],
              'smoke': [stage_smoke], 'all': [stage_prepare, stage_s1, stage_s2, stage_report]}[a.stage]
    for st in stages:
        st(a)


if __name__ == '__main__':
    main()

Writing cabu.py


In [ ]:
base = f'python cabu.py {{}} --data_root "{DATA_ROOT}" --work {WORK} {EXTRA}'
if RESUME_DIR:
    base += f' --resume_dir "{RESUME_DIR}"'
CMD = {st: base.format(st) for st in ('prepare', 's1', 's2', 'report')}
cmd_prepare, cmd_s1, cmd_s2, cmd_report = CMD['prepare'], CMD['s1'], CMD['s2'], CMD['report']
print(cmd_s1)
if RUN_PREPARE:
    !{cmd_prepare}

python cabu.py s1 --data_root "/kaggle/input/datasets/whats2000/breast-cancer-semantic-segmentation-bcss" --work /kaggle/working/cabu_run 
device: cuda; torch 2.11.0+cu130
found 48978 images and 44957 masks under /kaggle/input/datasets/whats2000/breast-cancer-semantic-segmentation-bcss; 44957 pairs carry a TCGA barcode
raw mask values in a sample of masks: [0, 1, 2, 3, 4, 9]


In [ ]:
if RUN_S1:
    !{cmd_s1}

In [ ]:
if RUN_S2:
    !{cmd_s2}

In [ ]:
if RUN_REPORT:
    !{cmd_report}
    from IPython.display import Image, display, Markdown
    rep = os.path.join(WORK, 'report')
    for f in sorted(glob.glob(os.path.join(rep, '*.png'))):
        print(os.path.basename(f)); display(Image(filename=f))

In [ ]:
# ---- package results to send back (models are left out to keep the zip small) -----------
pkg = '/kaggle/working/cabu_results'
shutil.rmtree(pkg, ignore_errors=True); os.makedirs(pkg)
for name in ('results.csv',):
    if os.path.exists(os.path.join(WORK, name)):
        shutil.copy(os.path.join(WORK, name), pkg)
for sub in ('report', 'traj'):
    if os.path.isdir(os.path.join(WORK, sub)):
        shutil.copytree(os.path.join(WORK, sub), os.path.join(pkg, sub))
shutil.copy(os.path.join(WORK, 'cache', 'meta.json'), os.path.join(pkg, 'cache_meta.json'))
shutil.make_archive('/kaggle/working/cabu_results', 'zip', pkg)
print('download /kaggle/working/cabu_results.zip')